# Text Preprocessing, Linguistic Normalization, and TF-IDF Representation

## 1. Academic Introduction & Motivation
Social media discourse is notorious for orthographic irregularity, informal syntax, intentional obfuscation, and multimodal tokens (emojis, hashtags, user handles). In cyberbullying detection, preprocessing involves a delicate trade-off:
* **Noise Reduction**: Stripping extraneous artifacts (HTTP URLs, HTML entities, platform-specific metadata) that inflate vocabulary sparsity without adding discriminative value.
* **Semantic Preservation**: Retaining negation markers (`not`, `never`, `no`), intensification punctuation, and cyberbullying slang that carry vital sentiment polarity.

### Methodological Objectives
1. Inspect concrete examples of noise in raw user-generated content.
2. Construct a modular, step-by-step cleaning pipeline covering URL removal, handle normalization, contraction expansion, and lemmatization.
3. Package the pipeline into a production-grade, reproducible `SocialMediaTextPreprocessor` class.
4. Partition the data into stratified training (80%) and hold-out testing (20%) sets.
5. Fit and parameterize a Term Frequency - Inverse Document Frequency (**TF-IDF**) vectorizer with sublinear scaling and n-gram windowing.
6. Persist processed datasets and the vectorizer artifact to `models/tfidf_vectorizer.joblib` for downstream classification models.

In [ ]:
# Import Scientific, Regular Expression, and NLP Libraries
import os
import re
import html
import string
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Output directory validation
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

LABEL_MAP = {0: 'Non-Bullying', 1: 'Bullying'}
print("Environment and directories configured successfully.")

## 2. Qualitative Inspection of Raw Social Media Noise
We load the raw corpus and examine characteristic instances of social media noise:
- Mentions (`@username`)
- Web links (`https://...`)
- HTML entity artifacts (`&amp;`, `&lt;`)
- Slang contractions (`don't`, `can't`, `u r`)
- Excessive repeated characters (`sooooooo`, `loser!!!!!`)

In [ ]:
# Ingest raw dataset
raw_data_path = os.path.join("data", "raw_dataset.csv")
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Raw dataset not found at '{raw_data_path}'. Please run Notebook 01 first.")

df_raw = pd.read_csv(raw_data_path)
print(f"Loaded raw dataset with {len(df_raw)} records.")

# Display diverse examples illustrating noise typologies
noise_examples = [
    "Hey @alex_99 check out this link: https://t.co/xyz123 &amp; read the article! #awesome",
    "You're an absolute l000ser and nobody likes uuuuu @user101 #toxic #hate",
    "RT @sammy: I can't believe they won't let us participate... It's totally unfair!!",
    "Shut the F*** UP already! You are SOOOO dumb... https://bit.ly/fake",
    "Enjoying some great coffee &amp; sunshine today! :) #blessed #weekend",
    "u r such an idiot, why don't you just leave the server already???",
    "Congratulations to @team_lead on winning the award!! Best leader EVER!",
    "Literal garbage human 🗑️🤮 delete your account right now @loser"
]

print("=== Exemplary Raw Social Media Texts ===")
for i, ex in enumerate(noise_examples, 1):
    print(f"[{i}] {ex}")

## 3. Modular Step-by-Step Preprocessing Components

We decompose the cleaning process into discrete, testable transformations:
1. **HTML & URL Normalization**: Strips URLs (`https?://\S+`) and decodes entities (`&amp;` $\rightarrow$ `&`).
2. **Social Mention & Hashtag Handling**: Removes direct `@user` handles; strips `#` symbol while retaining meaningful hashtag text.
3. **Contraction Expansion**: Expands contractions (`can't` $\rightarrow$ `cannot`, `you're` $\rightarrow$ `you are`) to ensure standard lexical matching.
4. **Character Flooding Normalization**: Truncates elongated sequences (`sooooo` $\rightarrow$ `soo`) to mitigate vocabulary inflation.
5. **Punctuation & Special Character Filtering**: Cleans extraneous punctuation while preserving word boundaries.
6. **Stopword Pruning with Negation Preservation**: Retains polarity-inverting terms (`not`, `no`, `never`, `neither`).

In [ ]:
# Comprehensive Contraction Expansion Dictionary
CONTRACTION_MAP = {
    "ain't": "am not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
    "'cause": "because", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "how'd": "how did", "how'll": "how will", "how's": "how is",
    "i'd": "i would", "i'll": "i will", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it'd": "it would", "it'll": "it will", "it's": "it is",
    "let's": "let us", "might've": "might have", "mightn't": "might not",
    "must've": "must have", "mustn't": "must not", "shan't": "shall not",
    "she'd": "she would", "she'll": "she will", "she's": "she is",
    "should've": "should have", "shouldn't": "should not", "that's": "that is",
    "there's": "there is", "they'd": "they would", "they'll": "they will",
    "they're": "they are", "they've": "they have", "wasn't": "was not",
    "we'd": "we would", "we'll": "we will", "we're": "we are", "we've": "we have",
    "weren't": "were not", "what'll": "what will", "what're": "what are",
    "what's": "what is", "what've": "what have", "where's": "where is",
    "who'll": "who will", "who's": "who is", "won't": "will not",
    "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
    "you're": "you are", "you've": "you have", "u": "you", "ur": "your",
    "r": "are", "im": "i am", "dont": "do not", "cant": "cannot", "wont": "will not"
}

# Negation Tokens Protected from Stopword Removal
PRESERVED_NEGATIONS = {
    'not', 'no', 'never', 'neither', 'nor', 'none', 'cannot', 'hardly',
    'scarcely', 'barely', 'without', 'against'
}

# Base Stopwords Set
BASE_STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your',
    'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she',
    'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their',
    'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that',
    'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an',
    'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of',
    'at', 'by', 'for', 'with', 'about', 'between', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in',
    'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once',
    'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'only', 'own',
    'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just',
    'should', 'now', 'rt'
}

CUSTOM_STOPWORDS = BASE_STOPWORDS - PRESERVED_NEGATIONS
print(f"Initialized stopword set with {len(CUSTOM_STOPWORDS)} words. Negation tokens protected: {len(PRESERVED_NEGATIONS)}.")

In [ ]:
# Step-by-Step Cleaning Demonstrations

def strip_html_and_urls(text):
    text = html.unescape(str(text))
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    return text

def clean_mentions_and_hashtags(text):
    text = re.sub(r'@\w+', '', text)            # Remove user mentions
    text = re.sub(r'#(\w+)', r'\1', text)       # Preserve hashtag content, strip '#'
    return text

def expand_contractions_fn(text):
    words = text.split()
    expanded = [CONTRACTION_MAP.get(w.lower(), w) for w in words]
    return ' '.join(expanded)

def reduce_repeated_chars(text):
    # Condense 3+ repeated characters down to 2 (e.g. 'sooooo' -> 'soo')
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def clean_special_chars(text):
    # Retain alphanumeric characters, spaces, and essential tokens
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Demonstrate transformations on sample string
sample_noisy = "RT @alex_99: I can't stand this sooooo annoying idiot!! Check: https://evil.com/leak #StopBullying &amp; win"
print("Raw:          ", sample_noisy)
t1 = strip_html_and_urls(sample_noisy)
print("HTML/URLs:    ", t1)
t2 = clean_mentions_and_hashtags(t1)
print("Handles/Tags: ", t2)
t3 = expand_contractions_fn(t2)
print("Contractions: ", t3)
t4 = reduce_repeated_chars(t3)
print("Char Flooding:", t4)
t5 = clean_special_chars(t4)
print("Cleaned:      ", t5)

## 4. Production-Ready Preprocessor Class
We encapsulate all components into `SocialMediaTextPreprocessor`, implementing a scikit-learn compatible interface with token-level stopword filtering and optional lemmatization.

In [ ]:
class SocialMediaTextPreprocessor:
    """
    Production-grade text preprocessor optimized for social media toxicity classification.
    """
    def __init__(self, remove_stopwords=True, min_word_len=2):
        self.remove_stopwords = remove_stopwords
        self.min_word_len = min_word_len
        self.stopwords = CUSTOM_STOPWORDS if remove_stopwords else set()

    def clean_text(self, text):
        if not isinstance(text, str):
            return ""
        
        # 1. Decode HTML entities and strip URLs
        text = html.unescape(text)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'<.*?>', '', text)
        
        # 2. Strip user handles and normalize hashtags
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'#(\w+)', r'\1', text)
        
        # 3. Lowercase
        text = text.lower()
        
        # 4. Expand contractions
        tokens = text.split()
        expanded_tokens = [CONTRACTION_MAP.get(t, t) for t in tokens]
        text = ' '.join(expanded_tokens)
        
        # 5. Character repetition compression
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
        
        # 6. Punctuation removal
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        
        # 7. Token filtering
        final_tokens = []
        for word in text.split():
            if len(word) >= self.min_word_len:
                if not (self.remove_stopwords and word in self.stopwords):
                    final_tokens.append(word)
                    
        return ' '.join(final_tokens)

    def transform(self, texts):
        return [self.clean_text(t) for t in texts]

# Instantiate preprocessor
preprocessor = SocialMediaTextPreprocessor(remove_stopwords=True)
print("SocialMediaTextPreprocessor initialized.")

## 5. Before-and-After Qualitative Case Studies
We process our eight exemplary noise strings and evaluate the before/after fidelity.

In [ ]:
comparison_rows = []
for ex in noise_examples:
    cleaned = preprocessor.clean_text(ex)
    comparison_rows.append({'Raw Input Text': ex, 'Cleaned Normalized Text': cleaned})

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option('display.max_colwidth', None)
print("=== Before/After Preprocessing Comparison Table ===")
print(comparison_df.to_string(index=False))

## 6. Corpus Transformation and Stratified Train/Test Partitioning

We apply the preprocessor to all samples in the dataset, remove records that become empty after stripping noise, and execute a stratified 80/20 train/test split.

In [ ]:
# Apply preprocessing to entire corpus
print(f"Preprocessing {len(df_raw)} records...")
df_clean = df_raw.copy()
df_clean['clean_text'] = df_clean['text'].apply(preprocessor.clean_text)

# Drop records that become empty post-cleaning
initial_count = len(df_clean)
df_clean = df_clean[df_clean['clean_text'].str.strip() != ''].reset_index(drop=True)
dropped = initial_count - len(df_clean)
print(f"Corpus cleaned. Retained {len(df_clean)} samples (dropped {dropped} empty records).")

# Stratified Train/Test Split (80% Train, 20% Test)
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_clean['label']
)

print(f"\nTraining partition:   {len(train_df):,} samples ({train_df['label'].mean():.2%} Bullying)")
print(f"Testing partition:    {len(test_df):,} samples ({test_df['label'].mean():.2%} Bullying)")

# Save partitions to disk
train_df.to_csv(os.path.join("data", "processed_train.csv"), index=False)
test_df.to_csv(os.path.join("data", "processed_test.csv"), index=False)
print("Persisted processed partitions to data/processed_train.csv and data/processed_test.csv.")

## 7. Statistical Vectorization via TF-IDF Representation

We represent textual documents numerically using **Term Frequency - Inverse Document Frequency (TF-IDF)**.
For term $t$ in document $d$ within corpus $D$:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$
where:
* **Sublinear TF Scaling**: $\text{TF}(t, d) = 1 + \log(\text{tf}(t, d))$ for $\text{tf} > 0$, dampening the effect of frequent repetitive tokens.
* **Smooth IDF**: $\text{IDF}(t, D) = \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$.
* **N-gram Range**: $(1, 2)$ captures unigrams ('idiot', 'disgusting') and contextual bigrams ('shut up', 'delete account', 'not good').
* **Feature Capping**: `max_features=10000` bounds dimensionality and prevents overfitting.

In [ ]:
# Configure and Fit TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    min_df=3,
    max_df=0.85
)

print("Fitting TF-IDF Vectorizer on training partition...")
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['clean_text'])

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f"Train TF-IDF Matrix Shape: {X_train_tfidf.shape}")
print(f"Test TF-IDF Matrix Shape:  {X_test_tfidf.shape}")

## 8. Vocabulary Topology, Sparsity, and Top Informative Features
We inspect the extracted vocabulary and compute matrix sparsity:
$$\text{Sparsity} = 100 \times \left(1 - \frac{\text{Non-Zero Elements}}{\text{Total Matrix Elements}}\right)$$

In [ ]:
# Vocabulary and Sparsity Diagnostics
vocab = tfidf_vectorizer.vocabulary_
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

unigrams = [w for w in feature_names if ' ' not in w]
bigrams = [w for w in feature_names if ' ' in w]

total_elements = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]
non_zero_elements = X_train_tfidf.nnz
sparsity = 100.0 * (1.0 - (non_zero_elements / total_elements))

print("=== TF-IDF Representation Topology ===")
print(f"Total Vocabulary Size:      {len(vocab):,}")
print(f"Unigrams Count:             {len(unigrams):,} ({len(unigrams)/len(vocab):.1%})")
print(f"Bigrams Count:              {len(bigrams):,} ({len(bigrams)/len(vocab):.1%})")
print(f"Non-Zero Entries in Train:  {non_zero_elements:,}")
print(f"Matrix Sparsity:            {sparsity:.4f}%")

In [ ]:
# Compute Mean TF-IDF Weights Across the Corpus
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top_indices = np.argsort(mean_tfidf)[::-1][:25]

top_features = feature_names[top_indices]
top_scores = mean_tfidf[top_indices]

# Plot Top 25 Highest Mean TF-IDF Features
plt.figure(figsize=(12, 6))
plt.barh(range(len(top_features)), top_scores, color='#2b5c8f', edgecolor='black', alpha=0.85)
plt.yticks(range(len(top_features)), top_features, fontsize=11)
plt.gca().invert_yaxis()
plt.title("Top 25 Informative TF-IDF Features (Mean Corpus Weight)", pad=12, fontweight='bold')
plt.xlabel("Mean TF-IDF Score")
plt.tight_layout()
plt.show()

## 9. Serialization of Preprocessing Artifacts
To ensure complete pipeline interoperability across model training modules and the production Flask application, we persist:
1. `models/tfidf_vectorizer.joblib`
2. `data/processed_train.csv`
3. `data/processed_test.csv`

In [ ]:
# Persist TF-IDF Vectorizer
vectorizer_path = os.path.join("models", "tfidf_vectorizer.joblib")
joblib.dump(tfidf_vectorizer, vectorizer_path)
print(f"TF-IDF Vectorizer successfully saved to: '{vectorizer_path}' ({os.path.getsize(vectorizer_path)/1024:.1f} KB)")

# Verification of saved artifact
loaded_vec = joblib.load(vectorizer_path)
test_sample = ["you are completely stupid and useless"]
test_transformed = loaded_vec.transform(test_sample)
print(f"Artifact verification passed: Output sparse shape = {test_transformed.shape}")